# Projeto 15 - Gestão de Cinema e Ingressos
**Disciplina:** Paradigmas de Linguagens de Programação | **Turma 3003** | **Semestre 2026.2**

**Equipe:** Luiz Jardel + Ana Rebeca

## Descrição do problema
Sistema de terminal para gerenciar sessões de cinema e a venda de ingressos, mantendo tudo em memória.
O sistema cadastra sessões e assentos, vende e cancela ingressos, impede venda duplicada de assento,
aplica preços por tipo (inteira, meia, vip) e gera a arrecadação por sessão.

## Como os dados circulam pelo programa
1. `PRECOS` (dicionário) guarda o valor de cada tipo de ingresso.
2. `sessoes` (lista) guarda todas as sessões. Cada sessão é um dicionário com `id`, `filme`, `horario`, `sala`,
   `assentos` (lista de códigos) e `ingressos` (lista de dicionários com `assento`, `tipo` e `preco`).
3. O menu lê a entrada do usuário (já validada) e chama as funções, passando a sessão como parâmetro.
4. As funções de venda e cancelamento alteram a lista `ingressos` da sessão. Assentos livres **não são guardados**:
   são calculados sempre a partir de `assentos` menos `ingressos`, então os dados nunca ficam inconsistentes.
5. Os relatórios percorrem `sessoes` e calculam tudo na hora, por isso os valores mudam conforme os dados.

In [ ]:
# ============ DADOS BASE ============
# Tabela de preços por tipo de ingresso
PRECOS = {"inteira": 30.0, "meia": 15.0, "vip": 45.0}

# Lista que guarda todas as sessões (cada sessão é um dicionário)
sessoes = []

: 

## Funções do sistema

In [ ]:
# ============ FUNÇÕES ============

def gerar_assentos(fileiras, por_fileira):
    """Gera os códigos dos assentos. Ex.: 2 fileiras x 3 -> ['A1','A2','A3','B1','B2','B3']"""
    letras = "ABCDEFGHIJ"
    lista = []
    for i in range(fileiras):
        for n in range(1, por_fileira + 1):
            lista.append(letras[i] + str(n))
    return lista


def criar_sessao(id_sessao, filme, horario, sala, assentos):
    """Monta e devolve o dicionário de uma sessão (faz o papel do construtor)."""
    return {
        "id": id_sessao,
        "filme": filme,
        "horario": horario,
        "sala": sala,
        "assentos": assentos,
        "ingressos": [],
    }


def proximo_id(lista):
    """Devolve o próximo identificador livre (maior id existente + 1)."""
    maior = 0
    for s in lista:
        if s["id"] > maior:
            maior = s["id"]
    return maior + 1


def buscar_sessao(lista, id_sessao):
    """Busca por identificador. Devolve o dicionário da sessão ou None se não existir."""
    for s in lista:
        if s["id"] == id_sessao:
            return s
    return None


def assentos_ocupados(sessao):
    """Devolve a lista de assentos já vendidos na sessão."""
    ocupados = []
    for ing in sessao["ingressos"]:
        ocupados.append(ing["assento"])
    return ocupados


def assentos_livres(sessao):
    """Devolve os assentos livres (calculado, nunca armazenado)."""
    ocupados = assentos_ocupados(sessao)
    livres = []
    for a in sessao["assentos"]:
        if a not in ocupados:
            livres.append(a)
    return livres


def vender_ingresso(sessao, assento, tipo):
    """Vende um ingresso. Devolve (True/False, mensagem). Trata os dois casos-limite."""
    assento = assento.strip().upper()
    tipo = tipo.strip().lower()

    if len(assentos_livres(sessao)) == 0:                      # caso-limite 1
        return False, "Sessão lotada: não há assentos livres."
    if tipo not in PRECOS:
        return False, "Tipo de ingresso inválido: " + tipo
    if assento not in sessao["assentos"]:
        return False, "Assento inexistente nesta sessão: " + assento
    if assento in assentos_ocupados(sessao):                   # caso-limite 2
        return False, "Assento já vendido: " + assento

    # Preço é guardado no ingresso: mudar a tabela depois não altera vendas antigas
    sessao["ingressos"].append({"assento": assento, "tipo": tipo, "preco": PRECOS[tipo]})
    return True, "Ingresso vendido: assento " + assento + " (" + tipo + ") R$ " + format(PRECOS[tipo], ".2f")


def cancelar_ingresso(sessao, assento):
    """Cancela o ingresso de um assento, liberando-o. Devolve (True/False, mensagem)."""
    assento = assento.strip().upper()
    for i in range(len(sessao["ingressos"])):
        if sessao["ingressos"][i]["assento"] == assento:
            sessao["ingressos"].pop(i)
            return True, "Ingresso do assento " + assento + " cancelado."
    return False, "Não há ingresso vendido para o assento " + assento + "."


def arrecadacao_sessao(sessao):
    """Soma o valor de todos os ingressos vendidos na sessão."""
    total = 0.0
    for ing in sessao["ingressos"]:
        total += ing["preco"]
    return total


def ocupacao_percentual(sessao):
    """Percentual de assentos ocupados. Evita divisão por zero."""
    total = len(sessao["assentos"])
    if total == 0:
        return 0.0
    return len(sessao["ingressos"]) / total * 100


def atualizar_sessao(sessao, campo, novo_valor):
    """Atualiza filme, horario ou sala. Os assentos/ingressos não são alterados (consistência)."""
    novo_valor = novo_valor.strip()
    if campo not in ("filme", "horario", "sala"):
        return False, "Campo inválido (use filme, horario ou sala)."
    if novo_valor == "":
        return False, "O novo valor não pode ser vazio."
    sessao[campo] = novo_valor
    return True, "Campo '" + campo + "' atualizado para: " + novo_valor


def relatorio_consolidado(lista):
    """Devolve um dicionário com contagens, percentuais e totais do cinema inteiro."""
    contagem = {}
    for tipo in PRECOS:
        contagem[tipo] = 0
    total_ingressos = 0
    total_assentos = 0
    arrecadacao = 0.0
    lotadas = 0
    for s in lista:
        total_assentos += len(s["assentos"])
        arrecadacao += arrecadacao_sessao(s)
        if len(assentos_livres(s)) == 0:
            lotadas += 1
        for ing in s["ingressos"]:
            contagem[ing["tipo"]] += 1
            total_ingressos += 1
    return {
        "contagem": contagem,
        "total_ingressos": total_ingressos,
        "total_assentos": total_assentos,
        "arrecadacao": arrecadacao,
        "lotadas": lotadas,
        "total_sessoes": len(lista),
    }


def ler_inteiro(mensagem):
    """Pede um número inteiro positivo e repete até a entrada ser válida."""
    while True:
        texto = input(mensagem).strip()
        if texto.isdigit() and int(texto) > 0:
            return int(texto)
        print("Entrada inválida. Digite um número inteiro maior que zero.")


def ler_texto(mensagem):
    """Pede um texto não vazio e repete até a entrada ser válida."""
    while True:
        texto = input(mensagem).strip()
        if texto != "":
            return texto
        print("Entrada inválida. O campo não pode ficar vazio.")

## Dados de exemplo (12 sessões cadastradas)

In [ ]:
# ============ DADOS INICIAIS: 12 sessões ============
# (id, filme, horário, sala, fileiras, assentos por fileira)
dados_iniciais = [
    (1,  "Interestelar",       "14:00", "Sala 1", 3, 4),
    (2,  "Interestelar",       "19:30", "Sala 1", 3, 4),
    (3,  "Divertida Mente 2",  "15:00", "Sala 2", 2, 5),
    (4,  "Divertida Mente 2",  "17:30", "Sala 2", 2, 5),
    (5,  "Duna: Parte Dois",   "20:00", "Sala 3", 3, 5),
    (6,  "Duna: Parte Dois",   "22:30", "Sala 3", 3, 5),
    (7,  "Homem-Aranha",       "16:00", "Sala 4", 2, 4),
    (8,  "Homem-Aranha",       "18:30", "Sala 4", 2, 4),
    (9,  "Matrix",             "21:00", "Sala 5", 2, 3),
    (10, "Coringa",            "19:00", "Sala 6", 2, 4),
    (11, "Barbie",             "13:30", "Sala 7", 2, 5),
    (12, "Pré-estreia",        "23:00", "Sala 8", 1, 3),   # sessão pequena: fácil de lotar
]

for (i, filme, hor, sala, fil, por) in dados_iniciais:
    sessoes.append(criar_sessao(i, filme, hor, sala, gerar_assentos(fil, por)))

# Vendas iniciais: id da sessão -> lista de (assento, tipo)
vendas_iniciais = {
    1: [("A1", "inteira"), ("A2", "meia"), ("B1", "vip")],
    2: [("A1", "inteira"), ("A2", "inteira"), ("A3", "meia"), ("B1", "vip"), ("C4", "inteira")],
    3: [("A1", "meia"), ("A2", "meia")],
    4: [("B3", "inteira")],
    5: [("A1", "vip"), ("A2", "vip"), ("A3", "inteira"), ("B1", "meia")],
    6: [("C5", "inteira")],
    7: [("A1", "inteira"), ("A2", "meia"), ("B2", "meia")],
    8: [],
    9: [("A1", "vip"), ("A2", "inteira")],
    10: [("A1", "inteira"), ("B4", "meia")],
    11: [("A5", "meia")],
    12: [("A1", "inteira"), ("A2", "meia"), ("A3", "vip")],   # sessão 12 já nasce lotada
}
for id_s in vendas_iniciais:
    sessao = buscar_sessao(sessoes, id_s)
    for (assento, tipo) in vendas_iniciais[id_s]:
        vender_ingresso(sessao, assento, tipo)

# Roteiro 1) estrutura inicial de dados
print("Tabela de preços:", PRECOS)
print("Total de sessões:", len(sessoes))
print("Exemplo de sessão (dicionário):")
print(sessoes[0])

## Testes (8 ou mais, com entrada, saída esperada e saída observada)
Os testes usam uma **sessão separada** (id 99, assentos A1 e A2), para não alterar os dados do sistema.
Eles rodam em sequência, pois cada um depende do estado deixado pelo anterior.

In [ ]:
# ============ TESTES ============
sessao_teste = criar_sessao(99, "Filme Teste", "10:00", "Sala T", ["A1", "A2"])
total_ok = 0
total_testes = 0

def testar(nome, entrada, esperado, observado):
    """Compara esperado x observado e imprime o resultado."""
    global total_ok, total_testes
    total_testes += 1
    passou = (esperado == observado)
    if passou:
        total_ok += 1
    print("Teste", total_testes, "-", nome)
    print("   Entrada :", entrada)
    print("   Esperado:", esperado)
    print("   Obtido  :", observado)
    print("   Resultado:", "PASSOU" if passou else "FALHOU")
    print()

ok, msg = vender_ingresso(sessao_teste, "A1", "inteira")
testar("Venda normal", "assento A1, inteira", True, ok)

ok, msg = vender_ingresso(sessao_teste, "A1", "meia")
testar("CASO-LIMITE: assento já vendido", "assento A1 de novo, meia", False, ok)

ok, msg = vender_ingresso(sessao_teste, "A2", "meia")
testar("Segunda venda normal (agora a sessão fica cheia)", "assento A2, meia", True, ok)

ok, msg = vender_ingresso(sessao_teste, "A1", "vip")
testar("CASO-LIMITE: sessão lotada", "qualquer venda com 0 assentos livres", False, ok)
testar("Mensagem de sessão lotada", "texto devolvido", True, "lotada" in msg)

testar("Arrecadação (30 + 15)", "sessão de teste", 45.0, arrecadacao_sessao(sessao_teste))
testar("Ocupação percentual", "2 de 2 assentos", 100.0, ocupacao_percentual(sessao_teste))

ok, msg = cancelar_ingresso(sessao_teste, "A1")
testar("Cancelamento normal", "assento A1", True, ok)
testar("Assento liberado após cancelar", "assentos livres", ["A1"], assentos_livres(sessao_teste))
testar("Arrecadação após cancelar (só a meia)", "sessão de teste", 15.0, arrecadacao_sessao(sessao_teste))

ok, msg = cancelar_ingresso(sessao_teste, "A1")
testar("Cancelar assento sem ingresso", "assento A1 de novo", False, ok)

ok, msg = vender_ingresso(sessao_teste, "A1", "gold")
testar("Tipo de ingresso inválido", "tipo gold", False, ok)

ok, msg = vender_ingresso(sessao_teste, "Z9", "inteira")
testar("Assento inexistente", "assento Z9", False, ok)

testar("Busca de sessão inexistente", "id 1234", None, buscar_sessao(sessoes, 1234))

print("RESULTADO FINAL:", total_ok, "de", total_testes, "testes passaram.")

## Menu interativo
Execute esta célula e use o menu. O campo para digitar aparece no topo da janela do VS Code.

In [ ]:
# ============ MENU ============

def listar_sessoes(lista):
    print("\nID | Filme | Horário | Sala | Livres/Total | Ocupação")
    for s in lista:
        livres = len(assentos_livres(s))
        print(s["id"], "|", s["filme"], "|", s["horario"], "|", s["sala"], "|",
              livres, "/", len(s["assentos"]), "|", format(ocupacao_percentual(s), ".1f") + "%")


def pedir_sessao(lista):
    """Pede um id e devolve a sessão, ou None se não existir."""
    id_s = ler_inteiro("ID da sessão: ")
    sessao = buscar_sessao(lista, id_s)
    if sessao is None:
        print("Sessão não encontrada.")
    return sessao


def menu_cadastrar():
    filme = ler_texto("Filme: ")
    horario = ler_texto("Horário (ex.: 19:30): ")
    sala = ler_texto("Sala: ")
    fileiras = ler_inteiro("Número de fileiras (1 a 10): ")
    while fileiras > 10:
        print("Máximo de 10 fileiras.")
        fileiras = ler_inteiro("Número de fileiras (1 a 10): ")
    por_fileira = ler_inteiro("Assentos por fileira: ")
    nova = criar_sessao(proximo_id(sessoes), filme, horario, sala, gerar_assentos(fileiras, por_fileira))
    sessoes.append(nova)
    print("Sessão cadastrada com ID", nova["id"], "e", len(nova["assentos"]), "assentos.")


def menu_vender():
    sessao = pedir_sessao(sessoes)
    if sessao is None:
        return
    print("Assentos livres:", assentos_livres(sessao))
    assento = ler_texto("Assento: ")
    print("Tipos:", PRECOS)
    tipo = ler_texto("Tipo (inteira/meia/vip): ")
    ok, msg = vender_ingresso(sessao, assento, tipo)
    print(msg)


def menu_cancelar():
    sessao = pedir_sessao(sessoes)
    if sessao is None:
        return
    print("Assentos ocupados:", assentos_ocupados(sessao))
    assento = ler_texto("Assento a cancelar: ")
    ok, msg = cancelar_ingresso(sessao, assento)
    print(msg)


def menu_buscar_atualizar():
    sessao = pedir_sessao(sessoes)
    if sessao is None:
        return
    print("Sessão:", sessao["filme"], "|", sessao["horario"], "|", sessao["sala"])
    print("Assentos livres:", assentos_livres(sessao))
    print("Ingressos vendidos:", sessao["ingressos"])
    resp = input("Deseja atualizar algum campo? (s/n): ").strip().lower()
    if resp == "s":
        campo = ler_texto("Campo (filme/horario/sala): ").lower()
        valor = ler_texto("Novo valor: ")
        ok, msg = atualizar_sessao(sessao, campo, valor)
        print(msg)


def menu_relatorios():
    print("\n--- Arrecadação por sessão ---")
    melhor = None
    for s in sessoes:
        valor = arrecadacao_sessao(s)
        print("Sessão", s["id"], "-", s["filme"], s["horario"], ": R$", format(valor, ".2f"),
              "| ocupação", format(ocupacao_percentual(s), ".1f") + "%")
        if melhor is None or valor > arrecadacao_sessao(melhor):
            melhor = s
    if melhor is not None:
        print("Sessão mais lucrativa:", melhor["filme"], melhor["horario"],
              "- R$", format(arrecadacao_sessao(melhor), ".2f"))

    print("\n--- Relatório consolidado ---")
    r = relatorio_consolidado(sessoes)
    print("Total de sessões:", r["total_sessoes"], "| lotadas:", r["lotadas"],
          "(" + format(r["lotadas"] / r["total_sessoes"] * 100 if r["total_sessoes"] else 0, ".1f") + "%)")
    print("Ingressos vendidos:", r["total_ingressos"], "de", r["total_assentos"], "assentos (",
          format(r["total_ingressos"] / r["total_assentos"] * 100 if r["total_assentos"] else 0, ".1f") + "%)")
    for tipo in r["contagem"]:
        qtd = r["contagem"][tipo]
        pct = qtd / r["total_ingressos"] * 100 if r["total_ingressos"] else 0
        print("  ", tipo, ":", qtd, "(" + format(pct, ".1f") + "%)")
    print("Arrecadação total: R$", format(r["arrecadacao"], ".2f"))


def menu():
    while True:
        print("\n===== CINEMA =====")
        print("1 - Listar sessões")
        print("2 - Cadastrar sessão")
        print("3 - Vender ingresso")
        print("4 - Cancelar ingresso")
        print("5 - Buscar / atualizar sessão")
        print("6 - Relatórios")
        print("0 - Sair")
        opcao = input("Escolha: ").strip()
        if opcao == "1":
            listar_sessoes(sessoes)
        elif opcao == "2":
            menu_cadastrar()
        elif opcao == "3":
            menu_vender()
        elif opcao == "4":
            menu_cancelar()
        elif opcao == "5":
            menu_buscar_atualizar()
        elif opcao == "6":
            menu_relatorios()
        elif opcao == "0":
            print("Encerrando o sistema. Até logo!")
            break
        else:
            print("Opção inválida. Escolha de 0 a 6.")

menu()

## Conclusão
(Escrevam aqui, com suas palavras: o que o sistema faz, o que aprenderam e as dificuldades encontradas.)

## Quadro de contribuição individual
| Integrante | Contribuições |
|---|---|
| Luiz Jardel | (preencher) |
| Ana Rebeca | (preencher) |